# Roadtrip Map — Colab Launcher

This notebook stays thin on purpose — all logic lives in `colab_build_script.py` in your GitHub repo.

**Run once per session:** Runtime → Run all  
**After editing script in VS Code:** push to GitHub, then re-run Cell 2 and Cell 4.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
print('\nTop-level folders in My Drive:')
for item in sorted(os.listdir('/content/drive/MyDrive')):
    if os.path.isdir(f'/content/drive/MyDrive/{item}'):
        print(f'  📁 {item}')

In [ ]:
# ── Cell 2: Get the build script from GitHub ──────────────────────────────────
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/roadtrip-map.git'  # ← set once
REPO_DIR    = '/content/roadtrip-map'

import os
if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    print('Cloning repo...')
    !git clone {GITHUB_REPO} {REPO_DIR}

!echo "Script version: $(git -C {REPO_DIR} log -1 --format='%h %s')"

In [ ]:
# ── Cell 3: Install dependencies (cached after first run) ─────────────────────
!pip install Pillow pillow-heif requests -q
print('Dependencies ready.')

In [ ]:
# ── Cell 4: Run the build ─────────────────────────────────────────────────────
# Edit TAKEOUT_PATH if your folder shortcut has a different name.
# Pass None (remove the value) to auto-scan your Drive for ZIPs.

TAKEOUT_PATH = '/content/drive/MyDrive/Takeout'  # ← adjust if needed
OUTPUT_DIR   = '/content/output'
CONFIG_PATH  = f'{REPO_DIR}/config.json'

!python {REPO_DIR}/colab_build_script.py \
    --takeout "{TAKEOUT_PATH}" \
    --output  "{OUTPUT_DIR}" \
    --config  "{CONFIG_PATH}"

In [ ]:
# ── Cell 5: Download results ──────────────────────────────────────────────────
from google.colab import files
import os

data_json  = f'{OUTPUT_DIR}/data.json'
photos_zip = f'{OUTPUT_DIR}/photos.zip'

if os.path.exists(data_json):
    files.download(data_json)
else:
    print('data.json not found — did the build succeed?')

if os.path.exists(photos_zip):
    size_mb = os.path.getsize(photos_zip) / 1e6
    print(f'Downloading photos.zip ({size_mb:.0f} MB)...')
    files.download(photos_zip)
else:
    print('photos.zip not found — did the build succeed?')